In [78]:
import yaml
import pandas as pd
import numpy as np
from datetime import time
import statsmodels.api as sm
import datetime as dt
from tqdm import tqdm
from typing import Literal



#df = pd.read_csv('data/SPY_15min_2020-01_to_2022-01.csv')


def load_config(path: str):
    """
    Loads the configuration from a YAML file.

    :param path: Path to the YAML configuration file.
    :return: A dictionary containing the configuration.
    """
    with open(path, 'r') as file:
        config = yaml.safe_load(file)
    return config

def load_df(type: Literal["news", "SPY"], config: dict):
    """
    Loads the DataFrame for the specified year from the configuration.

    :param year: The year for which to load the DataFrame.
    :param config: The configuration dictionary containing file paths.
    :return: A pandas DataFrame containing the data for the specified year.
    """
    year = config["year"]
    if type == "news":
        df = pd.read_csv(config["news_path"])
    else:
        try: df = pd.read_csv(config[year])
        except KeyError:
            raise ValueError(f"No file path configured for {year}.")
    return df

def to_ger_timestamp(string: str):
    """
    Converts a string in the format 'YYYY-MM-DD HH:MM:SS' to a timezone-aware timestamp in Europe/Berlin.

    :param string: A string representing a date and time in the format 'YYYY-MM-DD (optional: HH:MM:SS').
    :return: A pandas Timestamp object localized to Europe/Berlin timezone.
    """
    dt_obj = pd.to_datetime(string)
    return dt_obj.tz_localize("Europe/Berlin")

def prepare_spy_data(data: pd.DataFrame, filter_dates: dict):
    """
    Prepares the SPY data by converting the datetime column to a timezone-aware index,
    setting the frequency to 15 minutes, and filtering for the specified time range.

    Please note that in spite of the labeling, "eventstart_CET" is actually Europe7/Berlin as it changes between CET (UTC+1) and CEST (UTC+2).

    :param data: A pandas DataFrame containing the SPY data with a datetime column.
    :param filter_dates: A dictionary with 'start_date' and 'end_date' keys for filtering the data.
    :return: A pandas DataFrame with the prepared SPY data.
    """
    # 1. Timezone specific preparation
    data['datetime'] = pd.to_datetime(data['Unnamed: 0'])
    data['datetime'] = data['datetime'].dt.tz_localize('America/New_York')  # Localize to ET timezone
    data.set_index('datetime', inplace=True)
    data.index = pd.DatetimeIndex(data.index)  # Ensure time-aware index
    data = data.asfreq('15min')  # Ensure the index is at 15-minute frequency
    data.index = data.index.tz_convert("Europe/Berlin") # Ensures correct timezone
    data = data.between_time("15:30", "22:00")
    data = data[data.index.weekday < 5]  # Keep only Monday (0) through Friday (4)

    # 2. Other preparations
    data.drop(columns=['Unnamed: 0'], inplace=True)  # Drop the original datetime column to avoid interpolation issues
    data.interpolate(method='time', inplace=True)
    data["y^2"] = (data["close"].apply(lambda x: np.log(x)).diff()) ** 2
    data = data.loc[filter_dates["start_date"]:filter_dates["end_date"]]

     # 3. Dummies
    data["day_of_week"] = data.index.dayofweek
    data["hour_of_day"] = data.index.hour
    day_dummies = pd.get_dummies(data["day_of_week"], prefix="FE_day", drop_first=True)
    hour_dummies = pd.get_dummies(data["hour_of_day"], prefix="FE_hour", drop_first=True)
    data = pd.concat([data, day_dummies, hour_dummies], axis=1)

    return data.drop(columns=["open", "high", "low", "volume", "close", "day_of_week", "hour_of_day"])

config = load_config("config.yaml")
filter_dates = {"start_date": to_ger_timestamp(config["start_date"]), "end_date": to_ger_timestamp(config["end_date"])}
filter_dates.values()
df = prepare_spy_data(load_df("SPY", config), filter_dates)
df


,y^2,FE_day_1,FE_day_2,FE_day_3,FE_day_4,FE_hour_16,FE_hour_17,FE_hour_18,FE_hour_19,FE_hour_20,FE_hour_21,FE_hour_22
datetime,,,,,,,,,,,,
2020-01-03 15:30:00+01:00,4.245277e-05,False,False,False,True,False,False,False,False,False,False,False
2020-01-03 15:45:00+01:00,1.896942e-07,False,False,False,True,False,False,False,False,False,False,False
2020-01-03 16:00:00+01:00,3.487367e-08,False,False,False,True,True,False,False,False,False,False,False
2020-01-03 16:15:00+01:00,2.421707e-08,False,False,False,True,True,False,False,False,False,False,False
2020-01-03 16:30:00+01:00,1.399066e-06,False,False,False,True,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
2020-08-28 21:00:00+02:00,2.004112e-06,False,False,False,True,False,False,False,False,False,True,False
2020-08-28 21:15:00+02:00,7.707881e-07,False,False,False,True,False,False,False,False,False,True,False
2020-08-28 21:30:00+02:00,5.239030e-08,False,False,False,True,False,False,False,False,False,True,False


In [79]:
def safe_add(column: pd.Series, value: int): # Expects a datetimeIndex
    """
    Safely adds or subtracts a value in minutes to/from a datetime index, ensuring that the result remains within market open hours.
    This function is based on the internet appendix of the provided paper, thus, more information can be found there.

    :param column: A pandas Series with a DatetimeIndex, expected to be timezone-aware.
    :param value: An integer representing the number of minutes to add (positive) or subtract (negative).
    :return: A pandas Series with the same index, adjusted by the specified value in minutes.
    """
    # Confirm that the column has dates
    if not pd.api.types.is_datetime64_any_dtype(column):
        raise TypeError("Excpected a DateTime Series.")
    # Check if the index is timezone-aware
    if column.dt.tz is None:
        raise ValueError("DatetimeIndex must be timezone-aware.")
    # Cast to ET timezone for timezone aware consistency
    column = column.dt.tz_convert("America/New_York")
    # Check whether value is outside of market open hours
    hypothetical_values = column + pd.Timedelta(minutes=value)

    mask_before_open = (hypothetical_values.dt.time < dt.time(9,30))
    mask_during_open = (dt.time(9,30) <= hypothetical_values.dt.time) & (hypothetical_values.dt.time <= dt.time(16,0))
    mask_after_open = (hypothetical_values.dt.time > dt.time(16,0))

    date_of_action = column.dt.normalize() # Extract date and set hours to 0:00
    output = column.copy()
    output[mask_during_open] = output[mask_during_open] + pd.Timedelta(minutes=value)
    if value >= 0: # Safe add
        output[mask_before_open] = date_of_action[mask_before_open] + pd.Timedelta(hours=9, minutes=30) + pd.Timedelta(minutes=value)
        output[mask_after_open] = date_of_action[mask_after_open] + pd.Timedelta(days=1) + pd.Timedelta(hours=9, minutes=30) + pd.Timedelta(minutes=value)
    else: # Safe subtract
        # Window starts when trading closes
        output[mask_before_open] = date_of_action[mask_before_open] - pd.Timedelta(days=1) + pd.Timedelta(hours=16)
        output[mask_after_open] = date_of_action[mask_after_open] + pd.Timedelta(hours=16)

    return output # Caution: returns series with ET timezone




In [80]:
def prepare_news_df(data: pd.DataFrame):
    """
    Prepares the news DataFrame by converting the event start and end times to timezone-aware timestamps, filters the data based on the specified date range, and creates time windows for each event.

    :param data: A pandas DataFrame containing news events with 'eventstart_CET' and 'eventend_CET' columns.
    :return: A pandas DataFrame with the prepared news data, including 'window_start' and 'window_end' columns.
    """
    # Ensure correct timezone
    data["eventstart_CET"] = pd.to_datetime(data["eventstart_CET"], format="%d.%m.%Y %H:%M:%S").dt.tz_localize("Europe/Berlin", ambiguous=False)


    data["eventend_CET"] = pd.to_datetime(data["eventend_CET"], format="%d.%m.%Y %H:%M:%S").dt.tz_localize("Europe/Berlin")
    # Filter on timestamps of stock data
    data = data[(data["eventstart_CET"] >= filter_dates["start_date"]) & (data["eventstart_CET"] <= filter_dates["end_date"])]

    #data["eventstart_CET"] = data["eventstart_CET"].dt.floor("15min")  # Optional: Round down to the nearest 15 minutes (not mentioned in paper)

    # Correct time window creation according to paper:
    event_end_mask = data["eventend_CET"].notna()
    ad_hoc_mask = data["type"] == "Ad Hoc"

    # Only assign to rows where event_end_mask is False
    data.loc[~event_end_mask, "window_start"] = (
        safe_add(data.loc[~event_end_mask, "eventstart_CET"], -15)
        .dt.tz_convert("Europe/Berlin")
    )

    data.loc[~event_end_mask, "window_end"] = (
        safe_add(data.loc[~event_end_mask, "eventstart_CET"], 30)
        .dt.tz_convert("Europe/Berlin")
    )

    data.loc[ad_hoc_mask, "window_end"] = (
        safe_add(data.loc[ad_hoc_mask, "eventstart_CET"], 60)
        .dt.tz_convert("Europe/Berlin")
    )

    # Only assign to rows where event_end_mask is True
    data.loc[event_end_mask, "window_start"] = (
        safe_add(data.loc[event_end_mask, "eventstart_CET"], -20)
        .dt.tz_convert("Europe/Berlin")
    )

    data.loc[event_end_mask, "window_end"] = (
        safe_add(data.loc[event_end_mask, "eventend_CET"], 20)
        .dt.tz_convert("Europe/Berlin")
    )

    return data

df_news = prepare_news_df(load_df("news", config))

df_news

,eventstart,eventend,name,type,subtype,subsubtype,description,source,scheduled,eventstart_CET,eventend_CET,window_start,window_end
49275,02.01.2020 19:00:00,NaT,UK BRC Shop Price Index,Macro Release,UK,BRC Shop Price Index,NaN,Bloomberg,1,2020-01-03 01:00:00+01:00,NaT,2020-01-02 22:00:00+01:00,2020-01-03 16:00:00+01:00
49276,03.01.2020 02:45:00,NaT,FR CPI and Wages,Macro Release,FR,CPI and Wages,NaN,Bloomberg,1,2020-01-03 08:45:00+01:00,NaT,2020-01-02 22:00:00+01:00,2020-01-03 16:00:00+01:00
49277,03.01.2020 03:00:00,NaT,ES Unemployment Net,Macro Release,ES,Unemployment Net,NaN,Bloomberg,1,2020-01-03 09:00:00+01:00,NaT,2020-01-02 22:00:00+01:00,2020-01-03 16:00:00+01:00
49278,03.01.2020 04:00:00,NaT,DE CPI Hesse,Macro Release,DE,CPI Hesse,NaN,Bloomberg,1,2020-01-03 10:00:00+01:00,NaT,2020-01-02 22:00:00+01:00,2020-01-03 16:00:00+01:00
49279,03.01.2020 04:30:00,NaT,UK Monetary Aggregates,Macro Release,UK,Monetary Aggregates,NaN,Bloomberg,1,2020-01-03 10:30:00+01:00,NaT,2020-01-02 22:00:00+01:00,2020-01-03 16:00:00+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
51552,28.08.2020 05:00:00,NaT,IT Auction Result Bond or Note,Auction,IT Result,Bond or Note,NaN,Bloomberg,1,2020-08-28 11:00:00+02:00,NaT,2020-08-27 22:00:00+02:00,2020-08-28 16:00:00+02:00
51553,28.08.2020 09:05:00,28.08.2020 09:43:00,BoE: Speech by Governor,Central Bank,BoE,Speech by Governor,"Bailey at Jackson Hole: ""it looks from today’s...",https://www.bankofengland.co.uk/-/media/boe/fi...,0,2020-08-28 15:05:00+02:00,2020-08-28 15:43:00+02:00,2020-08-27 22:00:00+02:00,2020-08-28 16:03:00+02:00
51554,28.08.2020 09:45:00,NaT,US Chicago Purchasing Manager,Macro Release,US,Chicago Purchasing Manager,NaN,Bloomberg,1,2020-08-28 15:45:00+02:00,NaT,2020-08-28 15:30:00+02:00,2020-08-28 16:15:00+02:00
51555,28.08.2020 10:00:00,NaT,US University of Michigan Surveys,Macro Release,US,University of Michigan Surveys,NaN,Bloomberg,1,2020-08-28 16:00:00+02:00,NaT,2020-08-28 15:45:00+02:00,2020-08-28 16:30:00+02:00


In [81]:
def create_news_dummies(spy_df: pd.DataFrame, news_df: pd.DataFrame, group_by: str = "type"):
    """
    Creates dummy variables for news events based on the specified grouping level.

    :param spy_df: A pandas DataFrame containing SPY data with a datetime index.
    :param news_df: A pandas DataFrame containing news events with 'window_start' and 'window_end' columns.
    :param group_by: The column name to group by (e.g., 'subsubtype', 'type').
    :return: A pandas DataFrame with dummy variables for each unique value in the specified grouping column.
    """

    print(f"creating {len(news_df[group_by].unique())} event dummie variables")
    dummies = {}
    for subtype, df_type in news_df.groupby(group_by):
        mask = pd.Series(False, index=df.index)
        for _, row in df_type.iterrows():
            mask = mask | ((spy_df.index > row["window_start"]) & (spy_df.index <= row["window_end"]))
        dummies[f"D_{subtype}"] = mask

    return pd.concat([spy_df, pd.DataFrame(dummies)], axis=1)

df_complete = create_news_dummies(df, df_news, group_by="type")
df_complete

creating 4 event dummie variables


,y^2,FE_day_1,FE_day_2,FE_day_3,FE_day_4,FE_hour_16,FE_hour_17,FE_hour_18,FE_hour_19,FE_hour_20,FE_hour_21,FE_hour_22,D_Ad Hoc,D_Auction,D_Central Bank,D_Macro Release
datetime,,,,,,,,,,,,,,,,
2020-01-03 15:30:00+01:00,4.245277e-05,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True
2020-01-03 15:45:00+01:00,1.896942e-07,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True
2020-01-03 16:00:00+01:00,3.487367e-08,False,False,False,True,True,False,False,False,False,False,False,False,False,False,True
2020-01-03 16:15:00+01:00,2.421707e-08,False,False,False,True,True,False,False,False,False,False,False,False,False,False,True
2020-01-03 16:30:00+01:00,1.399066e-06,False,False,False,True,True,False,False,False,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-08-28 21:00:00+02:00,2.004112e-06,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False
2020-08-28 21:15:00+02:00,7.707881e-07,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False
2020-08-28 21:30:00+02:00,5.239030e-08,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False


In [82]:
# Regression
def run_regression(df: pd.DataFrame, window_amt: int = 4):
    """
    Runs an OLS regression on the provided DataFrame. Automatically handles the rolling variance calculation based on the specified window amount. Also extracts dummies and fixed effects from the DataFrame. Dummies need to start with "D_" and fixed effects with "FE_".

    :param df: A pandas DataFrame containing the data for regression.
    :param window_amt: The number of 15-minute intervals to use for the rolling variance calculation.
    :return: A fitted OLS model.
    """

    fixed_effects = df.filter(regex="^FE_").astype(float)
    lagged_variance = df['y^2'].rolling(window=window_amt).sum()  # assuming window_amt 15-min intervals according to paper
    event_dummies = df.filter(regex="^D_").astype(float)


    #x = pd.concat([event_dummies, fixed_effects, lagged_variance], axis=1)
    #x = pd.concat([event_dummies, fixed_effects, lagged_variance], axis=1).iloc[3:]

    x = event_dummies.iloc[k-1:]
    x = sm.add_constant(x)
    y = df["y^2"].iloc[k-1:].astype(float)

    return sm.OLS(y, x).fit()

model = run_regression(df_complete, window_amt=4)
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    y^2   R-squared:                       0.023
Model:                            OLS   Adj. R-squared:                  0.022
Method:                 Least Squares   F-statistic:                     27.09
Date:                Sat, 12 Jul 2025   Prob (F-statistic):           2.99e-22
Time:                        03:17:17   Log-Likelihood:                 34894.
No. Observations:                4614   AIC:                        -6.978e+04
Df Residuals:                    4609   BIC:                        -6.975e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const            7.639e-06   2.22e-06      3.448      0.001     3.3e-06     1.2e-05
D_Ad Hoc         6.523e-05   1.03e-05      6.319      0.000     4.5e-05    8.55e-05
D_Auction        1.026e-05   4.65e-06      2.205      0.028    1.14e-06    1.94e-05
D_Central Bank   3.571e-06   5.78e-06      0.618      0.537   -7.76e-06    1.49e-05
D_Macro Release  3.151e-05    6.1e-06      5.167      0.000    1.96e-05    4.35e-05
==============================================================================
Omnibus:                     9458.485   Durbin-Watson:                   1.957
Prob(Omnibus):                  0.000   Jarque-Bera (JB):         26451145.551
Skew:                          17.156   Prob(JB):                         0.00
Kurtosis:                     372.337   Cond. No.                         5.94
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [83]:
# Compute Omega_k
def compute_omega_k(model, df: pd.DataFrame, k: int):
    """
    Computes the Omega_k value based on the fitted OLS model and the DataFrame.

    :param model: A fitted OLS model from statsmodels.
    :param df: A pandas DataFrame containing the data with a column "y^2" representing the squared returns.
    :param k: Number of top k columns based on t-statistics.
    :return: The computed Omega_k value and the top k columns based on t-statistics.
    """
    event_columns = df.filter(regex="^D_").columns  # Select only event dummies
    tstats = model.tvalues[event_columns].abs()  # Absolute t-statistics
    top_k_columns = tstats.nlargest(k).index  # Select top k columns based on t-statistics
    betas = model.params[top_k_columns]

    # Formula from paper
    p_d_eq_1 = df[top_k_columns].mean()
    product = betas * p_d_eq_1
    mu_ysqr = np.mean(df["y^2"].dropna())  # Mean of y^2, excluding the first rows due to rolling window
    omega_k = sum(product) / mu_ysqr

    return omega_k, top_k_columns
k = 4 # Number of top news types to consider
omega, top_k_columns = compute_omega_k(model, df_complete, k)

In [84]:
#omega chatgpt version #TODO DAS LÖSCHEN WEIL HIER KOMMEN VIEL ZU GROßE WERTE RAUS HILFE
def compute_omega_i(df: pd.DataFrame, dummy: pd.Series):
    """
    Computes the impact on the variance for a given dummy variable in a DataFrame.

    :param: df: A pandas DataFrame containing the data with a column "y^2" representing the squared returns.
    :param: dummy: A string representing the name of the dummy variable column.
    :return: The computed Omega_i value.
    """
    dummy_col = df[dummy]
    ek_y2_given_D1 = df.loc[df[dummy] == 1, "y^2"].dropna().mean() # Accounting for first rows due to rolling window
    P_d1 = (df[dummy] == 1).mean()
    E_y2 = df["y^2"].dropna().mean()

    return (ek_y2_given_D1 * P_d1) / E_y2
print(compute_omega_i(df_complete, "D_Auction"))

def compute_omega_k(df: pd.DataFrame, k: int, model):
    """
    Computes the Omega_k value for the top k columns based on t-statistics.

    :param df: A pandas DataFrame containing the data with a column "y^2" representing the squared returns.
    :param k: Number of top k columns based on t-statistics.
    :param model: A fitted OLS model from statsmodels.
    :return: The computed Omega_k value.
    """
    event_dummies = df.filter(regex="^D_").columns  # Select only event dummies
    tstats = model.tvalues[event_dummies].abs()  # Absolute t-statistics
    top_k_columns = tstats.nlargest(k).index  # Select top k columns based on t-statistics
    print(top_k_columns)

    for dummy in top_k_columns:
        omega_i = compute_omega_i(df, dummy)
        print(f"Omega_{dummy}: {omega_i}")
compute_omega_k(df_complete, 4, model)

0.4432286705575838
Index(['D_Ad Hoc', 'D_Macro Release', 'D_Auction', 'D_Central Bank'], dtype='object')
Omega_D_Ad Hoc: 0.17845277629700373
Omega_D_Macro Release: 0.4385521111894742
Omega_D_Auction: 0.4432286705575838
Omega_D_Central Bank: 0.3178677961616748


In [85]:
#todo time series variance decomposition
#todo bar chart für Omega_k
#todo overnight dummies -> difference between innerhalb der Handelszeit & außerhalb der Handelszeit plotten !
#todo plots!



In [86]:
# Bootsrapping
def bootstrap(df: pd.DataFrame, n: int = 1000):
    """
    Generates bootstrap samples from the provided DataFrame.

    :param df: A pandas DataFrame to bootstrap from.
    :param n: Number of bootstrap samples to generate.
    :return: A generator yielding bootstrap samples.
    """

    for i in tqdm(range(n), desc="Bootstrapping"):
        sample = df.sample(frac=1, replace=True)
        yield sample

all_betas = []
all_tstats = []

for sampled_df in bootstrap(df_complete):
    model = run_regression(sampled_df, window_amt=4)
    all_betas.append(model.params[top_k_columns])
    all_tstats.append(model.tvalues[top_k_columns].abs())


Bootstrapping: 100%|██████████| 1000/1000 [00:03<00:00, 282.53it/s]


In [87]:
df_betas = pd.concat(all_tstats, axis=1)
df_betas.T.mean()


D_Ad Hoc           6.471168
D_Macro Release    5.279513
D_Auction          2.215554
D_Central Bank     1.306592
dtype: float64

In [88]:
# Comparing bootstrapped results to first regression results

In [89]:
df_tstats = pd.concat(all_betas, axis=1)
df_tstats.T.mean()


D_Ad Hoc           0.000066
D_Macro Release    0.000031
D_Auction          0.000010
D_Central Bank     0.000003
dtype: float64